# 01 — Il classificatore a feature con le etichette NewsGuard 2026 (tentativo 11)

Il TF-IDF + regressione logistica dei tentativi 06–10, invariato, con l'etichetta scelta da `ETICHETTA`: il livello
NewsGuard **2026** (l'obiettivo) o **2022** (il controllo). Stesso dataset e stessi fold (`2-dataset/01_dataset`).

- Nel training anche le testate di solo training, con il loro **peso** (un gruppo vale al più 5 testate); classi
  bilanciate sui pesi. Test e metriche solo sulle **296 testate valutate**.
- Fold **per gruppo** (reti ∪ owner ∪ Parent Domain del 2022 e del 2026). Vocabolario e IDF solo sul training del fold.
- **C** scelto tra 0,1, 1, 10 e 100 con una validazione interna a 3 fold per gruppi sul solo training (F1 macro per testata).
- Per testata: **voto di maggioranza** dei suoi articoli. Binario: affidabile = punteggio ≥ 60.
- Due baseline senza testo: maggioranza e scelta a caso con le proporzioni del training.

Varianti del weekend: con `TITOLO` accanto al TF-IDF dell'articolo c'è un **TF-IDF a parte sul titolo** (pulito dalla
cornice del sito; come nella Versione 4); con `SOLO_POLITICI` solo le testate con il solo argomento politico.

Si salva tutto (`<NOME>_<ETICHETTA>_fold_gruppo_articoli.csv`, `..._metriche.json`, `<NOME>_<ETICHETTA>_metriche.csv`,
`..._FATTO`) in `Results/3-classificazione/`.


In [1]:
import json
import time

import numpy as np
import scipy.sparse as sp
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, pesi_bilanciati, registratore, LIVELLI

In [2]:
NOME = "feature_tuned"
ETICHETTA = "2026"          # 2026 (obiettivo) / 2022 (controllo)
TITOLO = False             # True: anche il titolo (TF-IDF a parte, affiancato)
SOLO_POLITICI = False      # True: solo le testate con il solo argomento politico
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = "0.1,1,10,100" # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate (come la prima versione)
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: poche testate, risultati in Results/3-classificazione/prova/

In [3]:
# Parameters
ETICHETTA = 2026
TITOLO = True
SOLO_POLITICI = False
N_JOBS = 5


In [4]:
NOME = NOME + ("_titolo" if TITOLO else "") + ("_politici" if SOLO_POLITICI else "") + f"_{ETICHETTA}"
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(ETICHETTA, PROVA, SOLO_VALUTAZIONE, titolo=TITOLO, solo_politici=SOLO_POLITICI)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate ({art[art.valutazione].domain.nunique()} valutate), "
         f"taglio {TAGLIO_TOKEN or 'nessuno'}, solo valutate {SOLO_VALUTAZIONE} ===")

2026-10-23 14:59:12  === feature_tuned_titolo_2026: 71218 articoli, 835 testate (296 valutate), taglio nessuno, solo valutate False ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def matrici(a, b):
    """Il TF-IDF dell'articolo (e, con TITOLO, uno a parte sul titolo, affiancato): vocabolari e IDF solo su a."""
    v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
    if TITOLO:
        w = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=30_000, strip_accents="unicode")
        Xa, Xb = sp.hstack([Xa, w.fit_transform(a.titolo)]).tocsr(), sp.hstack([Xb, w.transform(b.titolo)]).tocsr()
    return Xa, Xb


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in str(C_GRIGLIA).split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a = tr[tr.domain.isin(testate.domain[itr])]
        b = tr[tr.domain.isin(testate.domain[iva]) & tr.valutazione]          # si valuta come nel test: solo testate valutate
        Xa, Xb = matrici(a, b)
        pa = pesi_bilanciati(a.label, a.peso)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, random_state=SEED).fit(Xa, a.label, sample_weight=pa).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[(art[schema] == fold) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    Xtr, X = matrici(tr, te)                                   # vocabolario e IDF: solo training; test: solo transform
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(Xtr, tr.label, sample_weight=pesi_bilanciati(tr.label, tr.peso))
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred = pred[art.valutazione.to_numpy()].reset_index(drop=True)      # solo le testate valutate
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-23 15:10:54  fold_gruppo fold 5: C scelto 1.0 (F1 interne 0.1: 0.318, 1.0: 0.326, 10.0: 0.293, 100.0: 0.273)


2026-10-23 15:11:06  fold_gruppo fold 1: C scelto 100.0 (F1 interne 0.1: 0.232, 1.0: 0.240, 10.0: 0.233, 100.0: 0.254)


2026-10-23 15:12:29  fold_gruppo fold 3: C scelto 0.1 (F1 interne 0.1: 0.403, 1.0: 0.322, 10.0: 0.293, 100.0: 0.264)


2026-10-23 15:13:49  fold_gruppo fold 4: C scelto 1.0 (F1 interne 0.1: 0.277, 1.0: 0.306, 10.0: 0.272, 100.0: 0.274)


2026-10-23 15:13:56  fold_gruppo fold 2: C scelto 0.1 (F1 interne 0.1: 0.334, 1.0: 0.320, 10.0: 0.294, 100.0: 0.275)


2026-10-23 15:13:59  fold_gruppo fold 5: fatto in 879 s (54298 training, 8216 test)


2026-10-23 15:14:49  fold_gruppo fold 1: fatto in 933 s (52715 training, 8243 test)


2026-10-23 15:15:09  fold_gruppo fold 3: fatto in 951 s (61084 training, 9150 test)


2026-10-23 15:16:25  fold_gruppo fold 2: fatto in 1028 s (55325 training, 7894 test)


2026-10-23 15:17:07  fold_gruppo fold 4: fatto in 1068 s (61450 training, 8987 test)


## Baseline senza testo

Per ogni fold, dalle testate valutate di training (come nella prima versione): la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [6]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.full(len(art), -1)
    for fold in sorted(art[schema].unique()):
        tr, te = art[(art[schema] != fold) & art.valutazione], art[(art[schema] == fold) & art.valutazione]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    b = pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})
    return b[art.valutazione.to_numpy()].reset_index(drop=True)


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}{'_politici' if SOLO_POLITICI else ''}_{ETICHETTA}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [7]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  \
fold       modello                                                            
per gruppo TF-IDF + LR (feature_tuned_titolo_2026)        0.402       0.359   
           baseline: maggioranza                          0.318       0.096   
           baseline: a caso                               0.240       0.223   

                                                    testata bin  \
fold       modello                                                
per gruppo TF-IDF + LR (feature_tuned_titolo_2026)        0.689   
           baseline: maggioranza                          0.601   
           baseline: a caso                               0.551   

                                                    testata tol0.5  \
fold       modello                                                   
per gruppo TF-IDF + LR (feature_tuned_titolo_2026)           0.576   
           baseline: maggioranza                             0.459   
           baseline: a caso                                  0.405   

                                                    testata tol1  \
fold       modello                                                 
per gruppo TF-IDF + LR (feature_tuned_titolo_2026)         0.750   
           baseline: maggioranza                           0.601   
           baseline: a caso                                0.571   

                                                    articolo 5cl  articolo bin  
fold       modello                                                              
per gruppo TF-IDF + LR (feature_tuned_titolo_2026)         0.303         0.645  
           baseline: maggioranza                           0.300         0.597  
           baseline: a caso                                0.250         0.551

In [8]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = LIVELLI
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,28,14,3,2,4
l2,11,32,8,11,5
l3,6,16,4,8,5
l4,9,18,4,44,19
l5,5,5,1,23,11


19